In [4]:
from typing import Optional, Dict, Any
import yfinance as yf
from datetime import datetime, date, timedelta
import pandas as pd

In [5]:
def get_historical_market_data(ticker: str, start_date: str, end_date: Optional[str] = None) -> Dict[str, Any]:
    """
    Fetches historical daily market data for a given stock ticker using the yfinance library.
    It returns a structured list of dictionaries, each containing a date and the closing price.

    Args:
        ticker (str): The stock ticker symbol (e.g., "NVDA").
        start_date (str): Start date of historical data to retrieve.
        end_date (str):  End date of historical data to retrieve.

    Returns:
        Dict[str, Any]: A dictionary containing the ticker and a list of historical data points,
                        or an error message if the data could not be fetched.
    """
    
    try:
        start_dt = datetime.strptime(start_date, '%Y-%m-%d').date()
        end_dt = datetime.strptime(end_date, '%Y-%m-%d').date() if end_date else date.today()

        if end_dt < start_dt:
            return {"error": f"Invalid date range: start_date {start_dt} is after end_date {end_dt}."}

        # yfinance end date is exclusive; add 1 day so the end date is included.
        end_exclusive = end_dt + timedelta(days=1)

        # Download data from yfinance
        stock_data = yf.download(
            ticker,
            start=start_dt.isoformat(),
            end=end_exclusive.isoformat(),
            progress=False
        )
        
        if stock_data.empty:
            return {
                "error": (
                    f"No data returned for ticker {ticker} between {start_dt} and {end_dt}. "
                    "Check the ticker symbol, date range, or network access to Yahoo Finance."
                )
            }
        
        # Create the structured response
        is_multi = isinstance(stock_data.columns, pd.MultiIndex)
        close_key = ("Close", ticker) if is_multi else "Close"
        volume_key = ("Volume", ticker) if is_multi else "Volume"

        historical_data = [
            {
                "date": index.strftime('%Y-%m-%d'),
                "close_price": round(row[close_key], 2),
                "volume": int(row[volume_key]) if not pd.isna(row[volume_key]) else 0
            }
            for index, row in stock_data.iterrows()
        ]
        
        return {"ticker": ticker, "historical_data": historical_data}
    except Exception as e:
        return {"error": f"An error occurred while fetching data for {ticker}: {str(e)}"}

In [6]:
import yfinance as yf
import numpy as np
from datetime import datetime, date, timedelta
from typing import List, Dict, Any, Optional, Union

import pandas as pd
import time
import random

_HISTORICAL_DATA_CACHE: Dict[str, Dict[str, Any]] = {}
# --- Tool 1: Real-World Data Fetching ---
def get_historical_market_data(ticker: str, start_date: str, end_date: Optional[str] = None) -> Dict[str, Any]:
    """
    Fetches historical daily market data for a given stock ticker using the yfinance library.
    It returns a structured list of dictionaries, each containing a date and the closing price.

    Args:
        ticker (str): The stock ticker symbol (e.g., "NVDA").
        start_date (str): Start date of historical data to retrieve.
        end_date (str):  End date of historical data to retrieve.

    Returns:
        Dict[str, Any]: A dictionary containing the ticker and a list of historical data points,
                        or an error message if the data could not be fetched.
    """
    
    def _cache_key(symbol: str, start_dt: date, end_dt: date) -> str:
        return f"{symbol.upper()}:{start_dt.isoformat()}:{end_dt.isoformat()}"

    def _is_rate_limit_error(err: Exception) -> bool:
        msg = str(err).lower()
        return "rate limit" in msg or "too many requests" in msg or "429" in msg

    def _download_with_backoff(symbol: str, start_dt: date, end_exclusive: date) -> pd.DataFrame:
        max_retries = 4
        base_backoff = 1.5
        last_err: Optional[Exception] = None

        for attempt in range(max_retries + 1):
            try:
                # Use a single-threaded request to reduce Yahoo's rate-limit pressure.
                return yf.download(
                    symbol,
                    start=start_dt.isoformat(),
                    end=end_exclusive.isoformat(),
                    interval="1d",
                    progress=False,
                    threads=False
                )
            except Exception as e:
                last_err = e
                if not _is_rate_limit_error(e) or attempt >= max_retries:
                    raise
                # Exponential backoff with jitter
                sleep_s = base_backoff * (2 ** attempt) + random.uniform(0, 0.5)
                time.sleep(sleep_s)

        if last_err:
            raise last_err
        raise RuntimeError("Unknown error while fetching data.")

    try:
        start_dt = datetime.strptime(start_date, '%Y-%m-%d').date()
        end_dt = datetime.strptime(end_date, '%Y-%m-%d').date() if end_date else date.today()

        if end_dt < start_dt:
            return {"error": f"Invalid date range: start_date {start_dt} is after end_date {end_dt}."}

        # yfinance end date is exclusive; add 1 day so the end date is included.
        end_exclusive = end_dt + timedelta(days=1)

        cache_key = _cache_key(ticker, start_dt, end_dt)
        if cache_key in _HISTORICAL_DATA_CACHE:
            return _HISTORICAL_DATA_CACHE[cache_key]

        # Download data from yfinance with backoff to mitigate rate limits
        stock_data = _download_with_backoff(ticker, start_dt, end_exclusive)
        
        if stock_data.empty:
            return {
                "error": (
                    f"No data returned for ticker {ticker} between {start_dt} and {end_dt}. "
                    "Check the ticker symbol, date range, or network access to Yahoo Finance."
                )
            }
        
        # Create the structured response
        is_multi = isinstance(stock_data.columns, pd.MultiIndex)
        close_key = ("Close", ticker) if is_multi else "Close"
        volume_key = ("Volume", ticker) if is_multi else "Volume"

        historical_data = [
            {
                "date": index.strftime('%Y-%m-%d'),
                "close_price": round(row[close_key], 2),
                "volume": int(row[volume_key]) if not pd.isna(row[volume_key]) else 0
            }
            for index, row in stock_data.iterrows()
        ]
        
        result = {"ticker": ticker, "historical_data": historical_data}
        _HISTORICAL_DATA_CACHE[cache_key] = result
        return result
    except Exception as e:
        return {"error": f"An error occurred while fetching data for {ticker}: {str(e)}"}

In [7]:
get_historical_market_data(ticker='MSFT', start_date='2025-01-01', end_date='2025-01-02')

/var/folders/h0/p8z9hpwd5qsdhvzrd2qb44k00000gn/T/ipykernel_14848/80827855.py:42: FutureWarning: YF.download() has changed argument auto_adjust default to True
  return yf.download(


{'ticker': 'MSFT',
 'historical_data': [{'date': '2025-01-02',
   'close_price': np.float64(414.57),
   'volume': 16896500}]}

In [3]:
def search_market_news(ticker: str, limit: int = 5) -> Dict[str, Any]:
    """
    Fetches the latest news headlines and summaries for a stock ticker.
    This allows the Agent to infer qualitative risks (regulatory, geopolitical, etc.)
    based on real-time events rather than hardcoded lists.

    Args:
        ticker (str): The stock ticker (e.g., "TSLA").
        limit (int): Number of news items to return.

    Returns:
        Dict[str, Any]: A list of news items with title, publisher, and link.
    """
    try:
        # yfinance has a .news attribute that fetches recent stories
        stock = yf.Ticker(ticker)
        news_items = stock.news
        
        if not news_items:
            return {"message": f"No recent news found for {ticker}."}
        
        formatted_news = []
        for item in news_items[:limit]:
            # Published timestamp conversion
            pub_time = datetime.fromtimestamp(item.get('providerPublishTime', 0)).strftime('%Y-%m-%d')
            formatted_news.append({
                "title": item.get('title'),
                "publisher": item.get('publisher'),
                "date": pub_time,
                "link": item.get('link')
            })
            
        return {
            "ticker": ticker,
            "latest_news": formatted_news,
            "instruction": "Use these headlines to infer qualitative risks (e.g. 'Lawsuit', 'Recall', 'Rates')."
        }
    except Exception as e:
        return {"error": f"Failed to fetch news: {str(e)}"}

In [4]:
search_market_news('TSLA', 5)

{'ticker': 'TSLA',
 'latest_news': [{'title': None,
   'publisher': None,
   'date': '1969-12-31',
   'link': None},
  {'title': None, 'publisher': None, 'date': '1969-12-31', 'link': None},
  {'title': None, 'publisher': None, 'date': '1969-12-31', 'link': None},
  {'title': None, 'publisher': None, 'date': '1969-12-31', 'link': None},
  {'title': None, 'publisher': None, 'date': '1969-12-31', 'link': None}],
 'instruction': "Use these headlines to infer qualitative risks (e.g. 'Lawsuit', 'Recall', 'Rates')."}

In [5]:
def get_historical_market_data(ticker: str, start_date: str, end_date: Optional[str] = None) -> Dict[str, Any]:
    """
    Fetches historical daily market data for a given stock ticker using the yfinance library.
    It returns a structured list of dictionaries, each containing a date and the closing price.

    Args:
        ticker (str): The stock ticker symbol (e.g., "NVDA").
        start_date (str): Start date of historical data to retrieve.
        end_date (str):  End date of historical data to retrieve.

    Returns:
        Dict[str, Any]: A dictionary containing the ticker and a list of historical data points,
                        or an error message if the data could not be fetched.
    """
    
    try:
        
        end_date = datetime.strptime(end_date, '%Y-%m-%d').date() if end_date else date.today()
        # Download data from yfinance
        stock_data = yf.download(ticker, start=start_date, end=end_date)
        
        if stock_data.empty:
            return {"error": f"No data found for ticker {ticker}. It may be an invalid symbol."}
        
        # Create the structured response
        historical_data = [
            {
                "date": date.strftime('%Y-%m-%d'),
                "close_price": round(row[('Close', ticker)], 2)
            }
            for date, row in stock_data.iterrows()
        ]
        
        return {"ticker": ticker, "historical_data": historical_data}
    except Exception as e:
        return {"error": f"An error occurred while fetching data for {ticker}: {str(e)}"}

In [6]:
get_historical_market_data('LLY', '2025-05-01', '2025-08-01')

/var/folders/h0/p8z9hpwd5qsdhvzrd2qb44k00000gn/T/ipykernel_4981/65535194.py:20: FutureWarning: YF.download() has changed argument auto_adjust default to True
  stock_data = yf.download(ticker, start=start_date, end=end_date)
[*********************100%***********************]  1 of 1 completed


{'ticker': 'LLY',
 'historical_data': [{'date': '2025-05-01', 'close_price': np.float64(789.58)},
  {'date': '2025-05-02', 'close_price': np.float64(818.93)},
  {'date': '2025-05-05', 'close_price': np.float64(816.78)},
  {'date': '2025-05-06', 'close_price': np.float64(770.71)},
  {'date': '2025-05-07', 'close_price': np.float64(772.3)},
  {'date': '2025-05-08', 'close_price': np.float64(747.17)},
  {'date': '2025-05-09', 'close_price': np.float64(730.39)},
  {'date': '2025-05-12', 'close_price': np.float64(751.27)},
  {'date': '2025-05-13', 'close_price': np.float64(741.81)},
  {'date': '2025-05-14', 'close_price': np.float64(711.49)},
  {'date': '2025-05-15', 'close_price': np.float64(729.12)},
  {'date': '2025-05-16', 'close_price': np.float64(754.62)},
  {'date': '2025-05-19', 'close_price': np.float64(752.35)},
  {'date': '2025-05-20', 'close_price': np.float64(744.28)},
  {'date': '2025-05-21', 'close_price': np.float64(722.3)},
  {'date': '2025-05-22', 'close_price': np.float64

In [8]:
datetime.now()

datetime.datetime(2025, 10, 20, 22, 26, 2, 486863)